# PathoScan — Benchmark Evaluation
Compares base Gemma 4 E4B vs PathoScan fine-tuned model on ISIC 2019 test set.

In [ ]:
import json, torch, pandas as pd, numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)
from unsloth import FastVisionModel

DATA_DIR = Path('/kaggle/input/isic-2019')
IMG_DIR  = DATA_DIR / 'ISIC_2019_Training_Input'

CLASSES = ['MEL','NV','BCC','AKIEC','BKL','DF','VASC','SCC']

# Load test set (200 samples, stratified)
gt_df = pd.read_csv(DATA_DIR / 'ISIC_2019_Training_GroundTruth.csv')
gt_df['label'] = gt_df[CLASSES].idxmax(axis=1)
test_df = gt_df.groupby('label').apply(lambda x: x.sample(min(25, len(x)), random_state=99)).reset_index(drop=True)
print(f'Test samples: {len(test_df)}')

In [ ]:
SYSTEM = 'You are PathoScan. Analyze the skin lesion and respond ONLY with valid JSON with key "condition_code" being one of: MEL, NV, BCC, AKIEC, BKL, DF, VASC, SCC.'
USER   = 'Analyze this skin lesion image and provide a triage assessment.'

def run_inference(model, tokenizer, img_path):
    img = Image.open(img_path).convert('RGB').resize((224,224))
    messages = [
        {'role':'system','content': SYSTEM},
        {'role':'user','content':[{'type':'image','image':img},{'type':'text','text':USER}]}
    ]
    inp = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(img, inp, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        return json.loads(text).get('condition_code','UNKNOWN')
    except:
        return 'PARSE_ERROR'

def evaluate_model(model, tokenizer, df, desc):
    preds, gts = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        img_path = IMG_DIR / f"{row['image']}.jpg"
        if not img_path.exists(): continue
        pred = run_inference(model, tokenizer, img_path)
        preds.append(pred); gts.append(row['label'])
    acc = accuracy_score(gts, preds)
    print(f'\nAccuracy: {acc:.3f}')
    print(classification_report(gts, preds, labels=CLASSES))
    return acc, gts, preds

In [ ]:
# --- Evaluate Base Model ---
print('=== BASE MODEL ===')
base_model, tokenizer = FastVisionModel.from_pretrained(
    'unsloth/gemma-4-e4b-it-unsloth-bnb-4bit', load_in_4bit=True)
FastVisionModel.for_inference(base_model)
base_acc, base_gt, base_pred = evaluate_model(base_model, tokenizer, test_df, 'Base model')

del base_model; torch.cuda.empty_cache()

In [ ]:
# --- Evaluate Fine-tuned Model ---
print('\n=== PATHOSCAN FINE-TUNED MODEL ===')
ft_model, tokenizer = FastVisionModel.from_pretrained(
    '/kaggle/working/pathoscan-lora', load_in_4bit=True)
FastVisionModel.for_inference(ft_model)
ft_acc, ft_gt, ft_pred = evaluate_model(ft_model, tokenizer, test_df, 'Fine-tuned')

print(f'\n--- IMPROVEMENT ---')
print(f'Base accuracy:       {base_acc:.3f}')
print(f'Fine-tuned accuracy: {ft_acc:.3f}')
print(f'Improvement:         +{(ft_acc - base_acc)*100:.1f}%')

In [ ]:
# Plot confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(ft_gt, ft_pred, labels=CLASSES)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES,
            cmap='Blues', ax=ax)
ax.set_title('PathoScan Confusion Matrix (Fine-tuned Gemma 4 E4B)')
ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')